## Import

In [28]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader
from PIL import Image
from pathlib import Path
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [17]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.images = sorted(self.image_dir.glob("*.*"))

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label_path = self.label_dir / (img_path.stem + ".txt")

        img = Image.open(img_path).convert("RGB")

        # Hämta bildens storlek innan to_tensor
        img_w, img_h = img.size

        img = torchvision.transforms.functional.to_tensor(img)

        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, w, h = map(float, line.split())

                    xmin = (x - w / 2) * img_w
                    ymin = (y - h / 2) * img_h
                    xmax = (x + w / 2) * img_w
                    ymax = (y + h / 2) * img_h

                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(cls) + 1)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        return img, target

    def __len__(self):
        return len(self.images)


dataset = Dataset("images", "labels")
print(len(dataset))

img, target = dataset[0]

print(img.shape)
print(target)

8450
torch.Size([3, 1080, 1920])
{'boxes': tensor([[1060.9728,   99.0900, 1126.7520,  213.8724]]), 'labels': tensor([1])}


In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

dataset = Dataset("images", "labels")

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

## Model

In [27]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(device)

cuda


In [37]:
optimizer = torch.optim.AdamW(model.parameters(), lr = 2e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=40, verbose=True) # Verbose is deprecated --> current_lrs = scheduler.get_last_lr()
Early_Stop_Patience = 100
epochs = 2

In [ ]:
model.train()

for epoch in range(epochs):
    total_loss = 0

    for imgs, targets in loader:

        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: {total_loss:.3f}")

Epoch 0: 139172.018
Epoch 1: 1271.449


KeyboardInterrupt: 